# Demo 3 - Learning Version: DuckLake Lakehouse for Document Classification

Welcome! This interactive notebook will teach you how to build a modern lakehouse architecture using DuckLake for managing document classifications and embeddings.

## Learning Objectives
By the end of this notebook, you will:
1. Understand lakehouse architecture concepts
2. Build a DuckLake lakehouse with proper schema design
3. Integrate document classification from Demo 2
4. Implement embeddings for semantic search
5. Create a RAG system with classification-aware filtering
6. Use advanced features like time travel and lineage tracking

## Key Concepts
- **Lakehouse**: Combines data lake flexibility with data warehouse features
- **DuckLake**: DuckDB extension providing lakehouse capabilities
- **Time Travel**: Query data as it existed at previous points in time
- **Row Lineage**: Track data transformations and provenance
- **ACID Transactions**: Ensure data consistency

## Prerequisites
- Understanding of Demo 2's classification system
- Basic knowledge of embeddings and vector search
- Familiarity with SQL and database concepts

Let's build a production-ready lakehouse! 🏗️

## Setup: Install and Import Dependencies

First, let's install and import all required libraries.

In [ ]:
# Install required packages
!pip install -q duckdb sentence-transformers docling langchain langchain-openai pydantic numpy

In [ ]:
import duckdb
import numpy as np
from datetime import datetime
from typing import Dict, List, Optional, Any, Tuple
from dataclasses import dataclass
import json
import os
from pathlib import Path

# Document processing
from docling.document_converter import DocumentConverter, PdfFormatOption, PowerpointFormatOption, WordFormatOption, HTMLFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling_core.types.doc import (
    TextItem, TableItem, PictureItem, ListItem,
    CodeItem, FormulaItem, SectionHeaderItem
)

# Embeddings
from sentence_transformers import SentenceTransformer

# Classification
from langchain_openai import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from enum import Enum

print("✅ All imports successful")

## Import Classification Schemas from Demo 2

We'll reuse the classification schemas from Demo 2.

In [ ]:
# Classification schemas from Demo 2
class SlideType(str, Enum):
    """Types of slides in a presentation"""
    TITLE = "title"
    CONTENT = "content"
    CHART = "chart"
    TABLE = "table"
    CONCLUSION = "conclusion"
    AGENDA = "agenda"
    REFERENCES = "references"
    QUESTIONS = "questions"

class ContentTheme(str, Enum):
    """Themes of content"""
    TECHNICAL = "technical"
    BUSINESS = "business"
    EDUCATIONAL = "educational"
    RESEARCH = "research"
    MARKETING = "marketing"
    GENERAL = "general"

class VisualDensity(str, Enum):
    """Visual density classification"""
    TEXT_HEAVY = "text_heavy"
    BALANCED = "balanced"
    VISUAL_HEAVY = "visual_heavy"
    MINIMAL = "minimal"

class SlideClassification(BaseModel):
    """Classification schema for a single slide"""
    
    slide_type: SlideType = Field(
        description="The primary type/purpose of this slide"
    )
    content_theme: ContentTheme = Field(
        description="The main theme or domain of the content"
    )
    visual_density: VisualDensity = Field(
        description="The visual density based on text vs visual elements ratio"
    )
    key_concepts: List[str] = Field(
        description="List of 3-5 key topics or concepts mentioned in the slide",
        min_items=1,
        max_items=5
    )
    has_code: bool = Field(
        description="Whether the slide contains code snippets"
    )
    has_formulas: bool = Field(
        description="Whether the slide contains mathematical formulas"
    )
    has_tables: bool = Field(
        description="Whether the slide contains tables"
    )
    has_figures: bool = Field(
        description="Whether the slide contains figures or images"
    )
    complexity_score: int = Field(
        description="Complexity score from 1 (very simple) to 10 (very complex). Must be an integer",
        ge=1,
        le=10
    )
    confidence_score: float = Field(
        description="Confidence in this classification from 0.0 to 1.0",
        ge=0.0,
        le=1.0
    )

class DocumentClassification(BaseModel):
    """Overall document classification"""
    
    document_type: str = Field(
        description="Type of document (presentation, paper, report, etc.)"
    )
    primary_theme: ContentTheme = Field(
        description="The dominant theme across the document"
    )
    target_audience: str = Field(
        description="Intended audience (students, professionals, researchers, etc.)"
    )
    overall_complexity: int = Field(
        description="Overall complexity from 1 to 10",
        ge=1,
        le=10
    )
    key_concepts: List[str] = Field(
        description="Main topics covered in the document"
    )

print("✅ Classification schemas imported")

## Task 1: Create DuckLake Lakehouse Infrastructure

### Learning Context
A lakehouse combines:
- **Data Lake**: Store raw data in open formats (Parquet)
- **Data Warehouse**: SQL queries, ACID transactions, schema enforcement
- **Additional Features**: Time travel, lineage tracking, schema evolution

DuckLake provides these capabilities on top of DuckDB.

### Your Task
Create a `DuckLakeLakehouse` class that:
1. Initializes DuckLake extension and connection
2. Creates tables for presentations, slides, embeddings, etc.
3. Implements ID generation and snapshot creation
4. Provides proper schema for classification data

### Hints
- Use `INSTALL ducklake; LOAD ducklake;` to enable extension
- Attach lakehouse with `ATTACH 'ducklake:path' AS name`
- Design tables with proper relationships (foreign keys)
- Store JSON data for flexible fields like key_concepts

In [ ]:
class DuckLakeLakehouse:
    def __init__(self, lakehouse_path: str = "./powerpoint_lakehouse.ducklake"):
        self.lakehouse_path = lakehouse_path
        self.conn = None
        self.setup_lakehouse()
        
    def setup_lakehouse(self):
        """Initialize DuckLake lakehouse with proper extension and connection"""
        # TODO: Step 1 - Create connection
        # self.conn = duckdb.connect(":memory:")
        
        # TODO: Step 2 - Install and load DuckLake extension
        # self.conn.execute("INSTALL ducklake;")
        # self.conn.execute("LOAD ducklake;")
        
        # TODO: Step 3 - Attach DuckLake database
        # self.conn.execute(f"ATTACH 'ducklake:{self.lakehouse_path}' AS ppt_lake;")
        
        # TODO: Step 4 - Create tables
        # self.create_tables()
        
        pass
        
    def create_tables(self):
        """Create lakehouse tables for PowerPoint analysis"""
        
        # TODO: Step 1 - Create presentations table
        # self.conn.execute("""
        # CREATE TABLE IF NOT EXISTS ppt_lake.presentations (
        #     id INTEGER,
        #     name VARCHAR,
        #     file_path VARCHAR,
        #     -- Classification fields from Demo 2
        #     document_type VARCHAR,
        #     primary_theme VARCHAR,
        #     target_audience VARCHAR,
        #     overall_complexity INTEGER,
        #     key_concepts VARCHAR,  -- JSON array
        #     -- Metadata
        #     created_timestamp BIGINT,
        #     slide_count INTEGER,
        #     processing_metadata VARCHAR  -- JSON
        # );
        # """)
        
        # TODO: Step 2 - Create slides table with classification
        # Think about what fields you need for slide classification
        
        # TODO: Step 3 - Create embeddings table
        # Consider: slide_id, chunk_text, embedding vector, chunk_index
        
        # TODO: Step 4 - Create tables for extracted content
        # slide_tables, slide_figures, content_elements
        
        # TODO: Step 5 - Create processing history table for lineage
        
        pass
        
    def get_next_id(self, table_name: str) -> int:
        """Get next ID for a table"""
        # TODO: Query max ID and return next value
        # result = self.conn.execute(f"""
        #     SELECT COALESCE(MAX(id), 0) + 1 as next_id 
        #     FROM ppt_lake.{table_name}
        # """).fetchone()
        # return result[0]
        
        return 1
        
    def create_snapshot(self, description: str):
        """Create a snapshot of the current lakehouse state"""
        # TODO: Record snapshot in processing history
        # timestamp = int(datetime.now().timestamp())
        # Insert into processing_history
        # Commit transaction
        # Return timestamp
        
        return int(datetime.now().timestamp())

# Create your lakehouse instance
# lakehouse = DuckLakeLakehouse()

### Solution for Task 1

In [ ]:
# SOLUTION
class DuckLakeLakehouse:
    def __init__(self, lakehouse_path: str = "./powerpoint_lakehouse.ducklake"):
        self.lakehouse_path = lakehouse_path
        self.conn = None
        self.setup_lakehouse()
        
    def setup_lakehouse(self):
        """Initialize DuckLake lakehouse with proper extension and connection"""
        # Create connection
        self.conn = duckdb.connect(":memory:")
        
        # Install DuckLake extension
        self.conn.execute("INSTALL ducklake;")
        self.conn.execute("LOAD ducklake;")
        
        # Attach DuckLake database
        self.conn.execute(f"ATTACH 'ducklake:{self.lakehouse_path}' AS ppt_lake;")
        
        # Create tables with proper DuckLake structure
        self.create_tables()
        
    def create_tables(self):
        """Create lakehouse tables for PowerPoint analysis"""
        
        # Presentations table
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS ppt_lake.presentations (
            id INTEGER,
            name VARCHAR,
            file_path VARCHAR,
            -- Classification fields from Demo 2
            document_type VARCHAR,
            primary_theme VARCHAR,
            target_audience VARCHAR,
            overall_complexity INTEGER,
            key_concepts VARCHAR,  -- JSON array
            -- Metadata
            created_timestamp BIGINT,
            slide_count INTEGER,
            processing_metadata VARCHAR  -- JSON
        );
        """)
        
        # Slides table with classification
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS ppt_lake.slides (
            id INTEGER,
            presentation_id INTEGER,
            slide_number INTEGER,
            title VARCHAR,
            content VARCHAR,
            -- Classification fields
            slide_type VARCHAR,
            content_theme VARCHAR,
            visual_density VARCHAR,
            complexity_score INTEGER,
            key_concepts VARCHAR,  -- JSON array
            -- Metadata
            created_timestamp BIGINT,
            has_tables INTEGER,
            has_figures INTEGER,
            table_count INTEGER,
            figure_count INTEGER
        );
        """)
        
        # Embeddings table
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS ppt_lake.embeddings (
            id INTEGER,
            slide_id INTEGER,
            chunk_text VARCHAR,
            chunk_index INTEGER,
            embedding VARCHAR,  -- JSON array
            created_timestamp BIGINT
        );
        """)
        
        # Tables table for table content
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS ppt_lake.slide_tables (
            id INTEGER,
            slide_id INTEGER,
            table_index INTEGER,
            table_content VARCHAR,  -- JSON
            row_count INTEGER,
            column_count INTEGER,
            created_timestamp BIGINT
        );
        """)
        
        # Figures table
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS ppt_lake.slide_figures (
            id INTEGER,
            slide_id INTEGER,
            figure_index INTEGER,
            figure_caption VARCHAR,
            figure_type VARCHAR,
            created_timestamp BIGINT
        );
        """)
        
        # Processing history table for lineage
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS ppt_lake.processing_history (
            id INTEGER,
            presentation_id INTEGER,
            processing_type VARCHAR,
            processing_timestamp BIGINT,
            processing_metadata VARCHAR  -- JSON
        );
        """)
        
    def get_next_id(self, table_name: str) -> int:
        """Get next ID for a table"""
        result = self.conn.execute(f"""
            SELECT COALESCE(MAX(id), 0) + 1 as next_id 
            FROM ppt_lake.{table_name}
        """).fetchone()
        return result[0]
        
    def create_snapshot(self, description: str):
        """Create a snapshot of the current lakehouse state"""
        # DuckLake automatically creates snapshots with each transaction
        # We can query specific snapshots using SNAPSHOT AS OF syntax
        timestamp = int(datetime.now().timestamp())
        self.conn.execute("""
            INSERT INTO ppt_lake.processing_history 
            VALUES (?, NULL, 'snapshot', ?, ?)
        """, [self.get_next_id('processing_history'), timestamp, json.dumps({"description": description})])
        self.conn.commit()
        return timestamp

# Initialize the lakehouse
lakehouse = DuckLakeLakehouse("./powerpoint_lakehouse.ducklake")
print("✅ DuckLake lakehouse initialized")

## Task 2: Integrate Document Processing with Classification

### Learning Context
We need to:
1. Extract content using Docling (like Demo 2)
2. Classify documents and slides
3. Generate embeddings for semantic search
4. Store everything in the lakehouse

### Your Task
Create a `LakehouseDocumentProcessor` that:
1. Uses Docling to extract and categorize content
2. Classifies slides and documents using LLMs
3. Generates embeddings for text chunks
4. Stores all data in the lakehouse with proper relationships

### Hints
- Reuse extraction logic from Demo 2
- Use SentenceTransformer for embeddings
- Chunk long text before embedding
- Store extracted tables and figures separately
- Track processing history for lineage

In [ ]:
class LakehouseDocumentProcessor:
    def __init__(self, lakehouse: DuckLakeLakehouse):
        self.lakehouse = lakehouse
        self.setup_processors()
        
    def setup_processors(self):
        # TODO: Step 1 - Configure Docling pipeline options
        # pipeline_options = PdfPipelineOptions(
        #     do_table_structure=True,
        #     do_picture_classification=True
        # )
        
        # TODO: Step 2 - Initialize DocumentConverter
        # self.converter = DocumentConverter(...)
        
        # TODO: Step 3 - Initialize embedding model
        # self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        
        # TODO: Step 4 - Initialize LLM for classification
        # self.llm = ChatOpenAI(temperature=0, model="gpt-4")
        
        pass
        
    def extract_and_classify(self, file_path: str) -> Dict[str, Any]:
        """Extract content from document and classify it using Docling categorization"""
        # TODO: Step 1 - Convert document using Docling
        # result = self.converter.convert(file_path)
        # doc = result.document
        
        # TODO: Step 2 - Initialize categorized content structure
        # content = {
        #     "metadata": self._extract_metadata(doc),
        #     "text_elements": [],
        #     "tables": [],
        #     # Add other categories...
        # }
        
        # TODO: Step 3 - Iterate through document elements
        # Use isinstance() to check item types
        # for item, level in doc.iterate_items():
        #     if isinstance(item, TextItem):
        #         # Add to text_elements
        #     elif isinstance(item, TableItem):
        #         # Add to tables
        #     # Handle other types...
        
        # TODO: Step 4 - Prepare slides and classify
        # slide_contents = self._prepare_slide_contents(content)
        # Classify each slide
        # Classify the document
        
        # TODO: Step 5 - Return structured result
        # return {
        #     "file_path": file_path,
        #     "extracted_content": content,
        #     "document_classification": document_classification,
        #     "slide_classifications": slide_classifications
        # }
        
        pass
    
    def process_to_lakehouse(self, file_path: str) -> int:
        """Process document and store in lakehouse"""
        # TODO: Step 1 - Extract and classify
        # doc_data = self.extract_and_classify(file_path)
        
        # TODO: Step 2 - Get next presentation ID
        # pres_id = self.lakehouse.get_next_id('presentations')
        
        # TODO: Step 3 - Insert presentation record
        # Extract classification data
        # Insert into presentations table
        
        # TODO: Step 4 - Process each slide
        # for idx, slide_class in enumerate(doc_data["slide_classifications"]):
        #     # Get slide ID
        #     # Insert slide record
        #     # Store tables and figures
        #     # Generate and store embeddings
        
        # TODO: Step 5 - Record processing history
        # Commit transaction
        # Return presentation ID
        
        return -1
    
    def chunk_text(self, text: str, max_length: int) -> List[str]:
        """Chunk text into smaller pieces"""
        # TODO: Implement text chunking
        # Split by words
        # Keep chunks under max_length
        # Return list of chunks
        
        return [text]

# Create your processor
# processor = LakehouseDocumentProcessor(lakehouse)

### Solution for Task 2 (Partial - Key Methods)

In [ ]:
# SOLUTION (Key methods shown)
class LakehouseDocumentProcessor:
    def __init__(self, lakehouse: DuckLakeLakehouse):
        self.lakehouse = lakehouse
        self.setup_processors()
        
    def setup_processors(self):
        # Configure pipeline options (from Demo 2)
        pipeline_options = PdfPipelineOptions(
            do_table_structure=True,
            do_picture_classification=True
        )
        
        # Use accurate TableFormer mode for better table extraction
        pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
        pipeline_options.table_structure_options.do_cell_matching = True
        
        # Initialize converter with multiple format support
        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
                InputFormat.PPTX: PowerpointFormatOption(pipeline_options=pipeline_options),
                InputFormat.DOCX: WordFormatOption(pipeline_options=pipeline_options),
                InputFormat.HTML: HTMLFormatOption(pipeline_options=pipeline_options)
            }
        )
        
        # Embedding model
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        
        # LLM for classification (using OpenAI as fallback)
        self.llm = ChatOpenAI(temperature=0, model="gpt-4")
    
    def chunk_text(self, text: str, max_length: int) -> List[str]:
        """Chunk text into smaller pieces"""
        words = text.split()
        chunks = []
        current_chunk = []
        current_length = 0
        
        for word in words:
            if current_length + len(word) + 1 > max_length:
                chunks.append(' '.join(current_chunk))
                current_chunk = [word]
                current_length = len(word)
            else:
                current_chunk.append(word)
                current_length += len(word) + 1
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks
    
    def classify_slide(self, title: str, content: str, table_count: int, figure_count: int,
                      has_code: bool, has_formulas: bool) -> SlideClassification:
        parser = PydanticOutputParser(pydantic_object=SlideClassification)
        
        prompt = PromptTemplate(
            template="""Analyze this presentation slide and classify it.

Slide Title: {title}
Slide Content: {content}
Number of Tables: {table_count}
Number of Figures: {figure_count}
Has Code: {has_code}
Has Formulas: {has_formulas}

{format_instructions}
""",
            input_variables=["title", "content", "table_count", "figure_count", "has_code", "has_formulas"],
            partial_variables={"format_instructions": parser.get_format_instructions()}
        )
        
        chain = prompt | self.llm | parser
        
        return chain.invoke({
            "title": title or "Untitled",
            "content": content[:1000] if content else "No content",
            "table_count": table_count,
            "figure_count": figure_count,
            "has_code": has_code,
            "has_formulas": has_formulas
        })

# Note: Full solution includes extract_and_classify, process_to_lakehouse, and other methods
# These are shown in the complete demo notebook

## Task 3: Build RAG System with Classification Filters

### Learning Context
RAG (Retrieval-Augmented Generation) systems need:
- **Semantic Search**: Find relevant content using embeddings
- **Metadata Filtering**: Filter by classification (theme, type, complexity)
- **Time Travel**: Query historical data
- **Context Generation**: Format results for LLM consumption

### Your Task
Create a `LakehouseRAGSystem` that:
1. Searches embeddings with similarity calculation
2. Applies classification filters (theme, type, complexity)
3. Supports time travel queries
4. Generates context from search results

### Hints
- Use cosine similarity for vector search
- Build SQL WHERE clause from filters
- Use `SNAPSHOT AS OF` for time travel
- Join embeddings, slides, and presentations tables

In [ ]:
class LakehouseRAGSystem:
    def __init__(self, lakehouse: DuckLakeLakehouse):
        self.lakehouse = lakehouse
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        
    def search_with_filters(self, 
                           query: str, 
                           top_k: int = 5,
                           theme_filter: Optional[str] = None,
                           slide_type_filter: Optional[str] = None,
                           min_complexity: Optional[int] = None,
                           max_complexity: Optional[int] = None,
                           presentation_id: Optional[int] = None,
                           snapshot_timestamp: Optional[int] = None) -> List[Dict[str, Any]]:
        """Search embeddings with classification filters and optional time travel"""
        
        # TODO: Step 1 - Generate query embedding
        # query_embedding = self.embedder.encode(query)
        
        # TODO: Step 2 - Build filter conditions
        # filters = []
        # if theme_filter:
        #     filters.append(f"s.content_theme = '{theme_filter}'")
        # if slide_type_filter:
        #     filters.append(f"s.slide_type = '{slide_type_filter}'")
        # Add other filters...
        # where_clause = " AND ".join(filters) if filters else "1=1"
        
        # TODO: Step 3 - Use time travel if snapshot provided
        # table_suffix = f" SNAPSHOT AS OF {snapshot_timestamp}" if snapshot_timestamp else ""
        
        # TODO: Step 4 - Build and execute query
        # query_sql = f"""
        # SELECT 
        #     e.id,
        #     e.slide_id,
        #     e.chunk_text,
        #     e.embedding,
        #     s.title,
        #     s.slide_type,
        #     s.content_theme,
        #     s.complexity_score,
        #     s.key_concepts,
        #     p.name as presentation_name,
        #     p.document_type,
        #     p.primary_theme
        # FROM ppt_lake.embeddings{table_suffix} e
        # JOIN ppt_lake.slides{table_suffix} s ON e.slide_id = s.id
        # JOIN ppt_lake.presentations{table_suffix} p ON s.presentation_id = p.id
        # WHERE {where_clause}
        # """
        
        # TODO: Step 5 - Calculate similarities
        # For each result:
        # - Parse embedding from JSON
        # - Calculate cosine similarity
        # - Store with metadata
        
        # TODO: Step 6 - Sort by similarity and return top_k
        
        return []
    
    def generate_context(self, search_results: List[Dict[str, Any]]) -> str:
        """Generate context from search results"""
        # TODO: Format search results into context string
        # Include document name, slide info, content, key concepts
        
        return ""

# Create your RAG system
# rag_system = LakehouseRAGSystem(lakehouse)

### Solution for Task 3

In [ ]:
# SOLUTION
class LakehouseRAGSystem:
    def __init__(self, lakehouse: DuckLakeLakehouse):
        self.lakehouse = lakehouse
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        
    def search_with_filters(self, 
                           query: str, 
                           top_k: int = 5,
                           theme_filter: Optional[str] = None,
                           slide_type_filter: Optional[str] = None,
                           min_complexity: Optional[int] = None,
                           max_complexity: Optional[int] = None,
                           presentation_id: Optional[int] = None,
                           snapshot_timestamp: Optional[int] = None) -> List[Dict[str, Any]]:
        """Search embeddings with classification filters and optional time travel"""
        
        # Generate query embedding
        query_embedding = self.embedder.encode(query)
        
        # Build filter conditions
        filters = []
        if theme_filter:
            filters.append(f"s.content_theme = '{theme_filter}'")
        if slide_type_filter:
            filters.append(f"s.slide_type = '{slide_type_filter}'")
        if min_complexity:
            filters.append(f"s.complexity_score >= {min_complexity}")
        if max_complexity:
            filters.append(f"s.complexity_score <= {max_complexity}")
        if presentation_id:
            filters.append(f"s.presentation_id = {presentation_id}")
            
        where_clause = " AND ".join(filters) if filters else "1=1"
        
        # Use time travel if snapshot timestamp provided
        table_suffix = f" SNAPSHOT AS OF {snapshot_timestamp}" if snapshot_timestamp else ""
        
        # Fetch all embeddings with filters
        query_sql = f"""
        SELECT 
            e.id,
            e.slide_id,
            e.chunk_text,
            e.embedding,
            s.title,
            s.slide_type,
            s.content_theme,
            s.complexity_score,
            s.key_concepts,
            p.name as presentation_name,
            p.document_type,
            p.primary_theme
        FROM ppt_lake.embeddings{table_suffix} e
        JOIN ppt_lake.slides{table_suffix} s ON e.slide_id = s.id
        JOIN ppt_lake.presentations{table_suffix} p ON s.presentation_id = p.id
        WHERE {where_clause}
        """
        
        results = self.lakehouse.conn.execute(query_sql).fetchall()
        
        # Calculate similarities
        similarities = []
        for row in results:
            db_embedding = np.array(json.loads(row[3]))
            similarity = np.dot(query_embedding, db_embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(db_embedding)
            )
            similarities.append({
                "similarity": similarity,
                "chunk_text": row[2],
                "slide_title": row[4],
                "slide_type": row[5],
                "content_theme": row[6],
                "complexity_score": row[7],
                "key_concepts": json.loads(row[8]) if row[8] else [],
                "presentation_name": row[9],
                "document_type": row[10],
                "primary_theme": row[11]
            })
        
        # Sort by similarity and return top_k
        similarities.sort(key=lambda x: x["similarity"], reverse=True)
        return similarities[:top_k]
    
    def generate_context(self, search_results: List[Dict[str, Any]]) -> str:
        """Generate context from search results"""
        context_parts = []
        for result in search_results:
            context_parts.append(f"""
Document: {result['presentation_name']} ({result['document_type']})
Slide: {result['slide_title']} (Type: {result['slide_type']}, Theme: {result['content_theme']})
Content: {result['chunk_text']}
Key Concepts: {', '.join(result['key_concepts'])}
---
""")
        return "\n".join(context_parts)

# Create RAG system
rag_system = LakehouseRAGSystem(lakehouse)
print("✅ RAG system initialized")

## Task 4: Implement Lakehouse Analytics

### Learning Context
Lakehouse analytics provide insights into:
- **Corpus Analysis**: Themes, types, complexity distribution
- **Data Lineage**: Track data transformations
- **Processing History**: Audit trail of operations
- **Snapshot Comparison**: Changes over time

### Your Task
Create a `LakehouseAnalytics` class that:
1. Analyzes theme distribution across corpus
2. Tracks data lineage for slides
3. Compares lakehouse snapshots
4. Provides processing history

### Hints
- Use GROUP BY for aggregations
- Join tables to get complete lineage
- Use SNAPSHOT AS OF for comparisons
- Return results as dictionaries or DataFrames

In [ ]:
class LakehouseAnalytics:
    def __init__(self, lakehouse: DuckLakeLakehouse):
        self.lakehouse = lakehouse
    
    def analyze_corpus_themes(self) -> Dict[str, Any]:
        """Analyze themes across all presentations"""
        # TODO: Step 1 - Query document type distribution
        # doc_types = self.lakehouse.conn.execute("""
        # SELECT document_type, COUNT(*) as count
        # FROM ppt_lake.presentations
        # GROUP BY document_type
        # ORDER BY count DESC
        # """).fetchall()
        
        # TODO: Step 2 - Query slide type distribution
        
        # TODO: Step 3 - Query content theme distribution
        
        # TODO: Step 4 - Calculate average complexity by type
        
        # TODO: Step 5 - Return structured results
        # return {
        #     "document_types": {row[0]: row[1] for row in doc_types},
        #     "slide_types": {...},
        #     "content_themes": {...},
        #     "avg_complexity_by_type": {...}
        # }
        
        return {}
    
    def get_data_lineage(self, slide_id: int) -> Dict[str, Any]:
        """Get complete data lineage for a slide"""
        # TODO: Step 1 - Get slide info with presentation
        # slide = self.lakehouse.conn.execute("""
        # SELECT s.*, p.name, p.file_path
        # FROM ppt_lake.slides s
        # JOIN ppt_lake.presentations p ON s.presentation_id = p.id
        # WHERE s.id = ?
        # """, [slide_id]).fetchone()
        
        # TODO: Step 2 - Get related tables, figures, embeddings
        
        # TODO: Step 3 - Build lineage structure
        
        return {}
    
    def compare_snapshots(self, timestamp1: int, timestamp2: int) -> Dict[str, Any]:
        """Compare lakehouse state between two snapshots"""
        # TODO: Count presentations at each snapshot
        # TODO: Find new presentations
        # TODO: Return comparison results
        
        return {}

# Create your analytics system
# analytics = LakehouseAnalytics(lakehouse)

### Solution for Task 4

In [ ]:
# SOLUTION
class LakehouseAnalytics:
    def __init__(self, lakehouse: DuckLakeLakehouse):
        self.lakehouse = lakehouse
    
    def get_presentation_history(self, presentation_id: int) -> List[Dict[str, Any]]:
        """Get processing history for a presentation"""
        results = self.lakehouse.conn.execute("""
        SELECT 
            processing_type,
            processing_timestamp,
            processing_metadata
        FROM ppt_lake.processing_history
        WHERE presentation_id = ?
        ORDER BY processing_timestamp DESC
        """, [presentation_id]).fetchall()
        
        return [{
            "type": row[0],
            "timestamp": row[1],
            "metadata": json.loads(row[2]) if row[2] else {}
        } for row in results]
    
    def analyze_corpus_themes(self) -> Dict[str, Any]:
        """Analyze themes across all presentations"""
        # Document type distribution
        doc_types = self.lakehouse.conn.execute("""
        SELECT document_type, COUNT(*) as count
        FROM ppt_lake.presentations
        GROUP BY document_type
        ORDER BY count DESC
        """).fetchall()
        
        # Slide type distribution
        slide_types = self.lakehouse.conn.execute("""
        SELECT slide_type, COUNT(*) as count
        FROM ppt_lake.slides
        GROUP BY slide_type
        ORDER BY count DESC
        """).fetchall()
        
        # Content theme distribution
        content_themes = self.lakehouse.conn.execute("""
        SELECT content_theme, COUNT(*) as count
        FROM ppt_lake.slides
        GROUP BY content_theme
        ORDER BY count DESC
        """).fetchall()
        
        # Average complexity by document type
        avg_complexity = self.lakehouse.conn.execute("""
        SELECT document_type, AVG(overall_complexity) as avg_complexity
        FROM ppt_lake.presentations
        GROUP BY document_type
        ORDER BY avg_complexity DESC
        """).fetchall()
        
        return {
            "document_types": {row[0]: row[1] for row in doc_types},
            "slide_types": {row[0]: row[1] for row in slide_types},
            "content_themes": {row[0]: row[1] for row in content_themes},
            "avg_complexity_by_type": {row[0]: row[1] for row in avg_complexity}
        }
    
    def get_data_lineage(self, slide_id: int) -> Dict[str, Any]:
        """Get complete data lineage for a slide"""
        # Get slide info
        slide = self.lakehouse.conn.execute("""
        SELECT s.*, p.name, p.file_path
        FROM ppt_lake.slides s
        JOIN ppt_lake.presentations p ON s.presentation_id = p.id
        WHERE s.id = ?
        """, [slide_id]).fetchone()
        
        if not slide:
            return {}
        
        # Get related data
        tables = self.lakehouse.conn.execute("""
        SELECT * FROM ppt_lake.slide_tables WHERE slide_id = ?
        """, [slide_id]).fetchall()
        
        figures = self.lakehouse.conn.execute("""
        SELECT * FROM ppt_lake.slide_figures WHERE slide_id = ?
        """, [slide_id]).fetchall()
        
        embeddings = self.lakehouse.conn.execute("""
        SELECT COUNT(*) as count, MIN(created_timestamp) as first_created
        FROM ppt_lake.embeddings WHERE slide_id = ?
        """, [slide_id]).fetchone()
        
        return {
            "slide_id": slide_id,
            "presentation_name": slide[15],
            "file_path": slide[16],
            "slide_number": slide[2],
            "classification": {
                "slide_type": slide[5],
                "content_theme": slide[6],
                "complexity_score": slide[8]
            },
            "derived_data": {
                "table_count": len(tables),
                "figure_count": len(figures),
                "embedding_count": embeddings[0],
                "first_processed": embeddings[1]
            }
        }
    
    def compare_snapshots(self, timestamp1: int, timestamp2: int) -> Dict[str, Any]:
        """Compare lakehouse state between two snapshots"""
        # Count presentations at each snapshot
        count1 = self.lakehouse.conn.execute(f"""
        SELECT COUNT(*) FROM ppt_lake.presentations SNAPSHOT AS OF {timestamp1}
        """).fetchone()[0]
        
        count2 = self.lakehouse.conn.execute(f"""
        SELECT COUNT(*) FROM ppt_lake.presentations SNAPSHOT AS OF {timestamp2}
        """).fetchone()[0]
        
        # Find new presentations
        new_presentations = self.lakehouse.conn.execute(f"""
        SELECT name FROM ppt_lake.presentations SNAPSHOT AS OF {timestamp2}
        WHERE id NOT IN (
            SELECT id FROM ppt_lake.presentations SNAPSHOT AS OF {timestamp1}
        )
        """).fetchall()
        
        return {
            "snapshot1_count": count1,
            "snapshot2_count": count2,
            "presentations_added": count2 - count1,
            "new_presentation_names": [row[0] for row in new_presentations]
        }

# Create analytics system
analytics = LakehouseAnalytics(lakehouse)
print("✅ Analytics system initialized")

## Challenge Task: Complete Lakehouse Pipeline

### Your Challenge
Build a complete pipeline that:

1. Processes multiple documents in batch
2. Creates snapshots at key points
3. Performs RAG searches with filters
4. Analyzes the corpus
5. Demonstrates time travel capabilities

### Requirements
- Handle errors gracefully
- Show progress updates
- Create meaningful snapshots
- Generate analysis reports
- Export results

### Starter Code
Build your solution below:

In [ ]:
async def lakehouse_pipeline_demo(
    input_files: List[str],
    search_queries: List[str],
    output_dir: str = "./lakehouse_results"
):
    """
    Complete lakehouse pipeline demonstration.
    
    Args:
        input_files: List of document files to process
        search_queries: List of queries to test RAG
        output_dir: Directory for results
    
    Returns:
        Pipeline execution report
    """
    # TODO: Your implementation here
    # Steps:
    # 1. Initialize lakehouse and processors
    # 2. Create initial snapshot
    # 3. Process documents in batch
    # 4. Create post-processing snapshot
    # 5. Perform RAG searches with different filters
    # 6. Analyze corpus themes
    # 7. Compare snapshots
    # 8. Export results and reports
    
    pass

# Test your pipeline
# sample_files = [
#     "/path/to/doc1.pptx",
#     "/path/to/doc2.pdf"
# ]
# queries = [
#     "machine learning concepts",
#     "data visualization techniques"
# ]
# report = await lakehouse_pipeline_demo(sample_files, queries)

### Solution for Challenge Task

In [ ]:
# SOLUTION
async def lakehouse_pipeline_demo(
    input_files: List[str],
    search_queries: List[str],
    output_dir: str = "./lakehouse_results"
):
    """
    Complete lakehouse pipeline demonstration.
    """
    import os
    from datetime import datetime
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    print("🚀 Starting Lakehouse Pipeline Demo")
    
    # Step 1: Initialize components
    print("\n📦 Initializing lakehouse components...")
    lakehouse = DuckLakeLakehouse(os.path.join(output_dir, "demo_lakehouse.ducklake"))
    processor = LakehouseDocumentProcessor(lakehouse)
    rag_system = LakehouseRAGSystem(lakehouse)
    analytics = LakehouseAnalytics(lakehouse)
    
    # Step 2: Create initial snapshot
    snapshot1 = lakehouse.create_snapshot("Initial empty lakehouse")
    print(f"📸 Created initial snapshot: {snapshot1}")
    
    # Step 3: Process documents
    print(f"\n📄 Processing {len(input_files)} documents...")
    processed_ids = []
    
    for idx, file_path in enumerate(input_files):
        if os.path.exists(file_path):
            print(f"  Processing {idx+1}/{len(input_files)}: {os.path.basename(file_path)}")
            try:
                pres_id = processor.process_to_lakehouse(file_path)
                processed_ids.append(pres_id)
                print(f"    ✅ Stored with ID: {pres_id}")
            except Exception as e:
                print(f"    ❌ Error: {str(e)}")
        else:
            print(f"    ⚠️ File not found: {file_path}")
    
    # Step 4: Create post-processing snapshot
    snapshot2 = lakehouse.create_snapshot(f"After processing {len(processed_ids)} documents")
    print(f"\n📸 Created post-processing snapshot: {snapshot2}")
    
    # Step 5: Perform RAG searches
    print("\n🔍 Testing RAG searches...")
    search_results = {}
    
    for query in search_queries:
        print(f"\n  Query: '{query}'")
        
        # Search without filters
        results = rag_system.search_with_filters(query, top_k=3)
        print(f"    Found {len(results)} results (no filters)")
        
        # Search with theme filter
        technical_results = rag_system.search_with_filters(
            query, theme_filter="technical", top_k=3
        )
        print(f"    Found {len(technical_results)} results (technical theme)")
        
        # Search with complexity filter
        complex_results = rag_system.search_with_filters(
            query, min_complexity=7, top_k=3
        )
        print(f"    Found {len(complex_results)} results (complexity >= 7)")
        
        search_results[query] = {
            "all": results,
            "technical": technical_results,
            "complex": complex_results
        }
    
    # Step 6: Analyze corpus
    print("\n📊 Analyzing corpus...")
    corpus_analysis = analytics.analyze_corpus_themes()
    
    print("  Document Types:", corpus_analysis.get("document_types", {}))
    print("  Content Themes:", corpus_analysis.get("content_themes", {}))
    print("  Slide Types:", list(corpus_analysis.get("slide_types", {}).keys())[:5])
    
    # Step 7: Compare snapshots
    print("\n🔄 Comparing snapshots...")
    snapshot_comparison = analytics.compare_snapshots(snapshot1, snapshot2)
    print(f"  Documents added: {snapshot_comparison['presentations_added']}")
    print(f"  New presentations: {snapshot_comparison['new_presentation_names']}")
    
    # Step 8: Time travel query
    print("\n⏰ Testing time travel...")
    if search_queries:
        # Search at initial snapshot (should be empty)
        historical_results = rag_system.search_with_filters(
            search_queries[0], 
            snapshot_timestamp=snapshot1,
            top_k=3
        )
        print(f"  Results at snapshot1: {len(historical_results)} (should be 0)")
        
        # Search at current state
        current_results = rag_system.search_with_filters(
            search_queries[0],
            top_k=3
        )
        print(f"  Results at current: {len(current_results)}")
    
    # Step 9: Export results
    print("\n💾 Exporting results...")
    
    # Export corpus analysis
    with open(os.path.join(output_dir, "corpus_analysis.json"), 'w') as f:
        json.dump(corpus_analysis, f, indent=2)
    
    # Export search results
    with open(os.path.join(output_dir, "search_results.json"), 'w') as f:
        # Convert search results to serializable format
        serializable_results = {}
        for query, results_dict in search_results.items():
            serializable_results[query] = {}
            for key, results in results_dict.items():
                serializable_results[query][key] = [
                    {k: v for k, v in r.items() if k != "similarity" or isinstance(v, (str, int, float, list, dict))}
                    for r in results
                ]
        json.dump(serializable_results, f, indent=2)
    
    # Create summary report
    report = {
        "pipeline_run": datetime.now().isoformat(),
        "lakehouse_path": lakehouse.lakehouse_path,
        "documents_processed": len(processed_ids),
        "successful_ids": processed_ids,
        "snapshots": {
            "initial": snapshot1,
            "post_processing": snapshot2
        },
        "corpus_summary": {
            "total_presentations": sum(corpus_analysis.get("document_types", {}).values()),
            "themes": list(corpus_analysis.get("content_themes", {}).keys()),
            "avg_complexity": corpus_analysis.get("avg_complexity_by_type", {})
        },
        "search_queries_tested": len(search_queries),
        "output_files": [
            "corpus_analysis.json",
            "search_results.json",
            "pipeline_report.json"
        ]
    }
    
    with open(os.path.join(output_dir, "pipeline_report.json"), 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"\n✅ Pipeline complete! Results saved to: {output_dir}")
    return report

# Example usage:
# sample_files = [
#     "/Users/daviddrummond/coursera/raw ppt files/MIT/0acb1321504f570d8ac581770f0ac5b5_MIT18_404f20_lec21.pptx",
#     "/Users/daviddrummond/coursera/raw ppt files/MIT/2caf9c2883b20fc2afdfa3cf42ae2ed0_MIT18_404f20_lec26.pptx"
# ]
# queries = [
#     "machine learning concepts",
#     "computational complexity"
# ]
# report = await lakehouse_pipeline_demo(sample_files, queries)

## Summary & Next Steps

### What You've Learned
✅ Built a proper lakehouse architecture with DuckLake  
✅ Integrated document classification from Demo 2  
✅ Implemented embeddings and semantic search  
✅ Created RAG with classification-aware filtering  
✅ Used advanced features: time travel, lineage, snapshots  
✅ Built analytics for corpus insights  

### Key Lakehouse Concepts Mastered
1. **Open Table Format**: Data stored as Parquet with SQL metadata
2. **ACID Transactions**: Ensuring data consistency
3. **Time Travel**: Query historical data states
4. **Schema Evolution**: Add columns without rewriting data
5. **Row Lineage**: Track data transformations
6. **Snapshot Isolation**: Consistent views of data

### Practice Exercises
1. **Add Data Quality Checks**: Validate classifications before storage
2. **Implement Incremental Processing**: Only process new documents
3. **Add Vector Indexes**: Speed up similarity searches
4. **Create Data Governance**: Track who processed what and when
5. **Build REST API**: Expose lakehouse capabilities as services

### Performance Considerations
- **Parallelization**: Process multiple documents concurrently
- **Batch Operations**: Group database operations
- **Caching**: Cache embeddings and classifications
- **Partitioning**: Partition data by date or theme

### Next Steps
- **Scale Up**: Process thousands of documents
- **Add ML Pipelines**: Train models on lakehouse data
- **Build Dashboards**: Visualize corpus analytics
- **Implement Data Mesh**: Federate across departments

### Resources
- [DuckDB Documentation](https://duckdb.org/docs/)
- [Lakehouse Architecture](https://www.databricks.com/glossary/data-lakehouse)
- [Vector Databases](https://www.pinecone.io/learn/vector-database/)
- [RAG Best Practices](https://www.anyscale.com/blog/a-comprehensive-guide-for-building-rag-based-llm-applications-part-1)

Congratulations on building a production-ready lakehouse! 🎉